# Gogoberidze, Kahniashvili, Kosowsky (2007)
## Stationary MHD Turbulence — GW Spectrum Workflow

**Reference**: Gogoberidze, Kahniashvili & Kosowsky, *The spectrum of gravitational radiation from primordial turbulence*, Phys. Rev. D **76**, 123504 (2007)

---

### Goal
Reproduce Fig. 1 of the paper: the gravitational-wave strain amplitude $h_c(f)$ from **stationary** isotropic Kolmogorov MHD turbulence, comparing the exact double-integral result to the aeroacoustic ($p\to 0$) approximation.

### Physical picture
- At a first-order phase transition, bubble collisions drive turbulence on a scale $l_0 = v_b \beta^{-1}$.  
- As long as the Hubble horizon is much larger than $l_0$, the turbulence looks **stationary** to GW observers.  
- The equal-time stress-energy spectrum is Kolmogorov: $E(k) \propto k^{-5/3}$ for $k \geq k_0 = 2\pi/l_0$.  
- Temporal decorrelation is modelled with the Kraichnan DIA (random sweeping): the unequal-time two-point correlator falls off as a Gaussian with rate $\eta(k) \propto (k^3 E(k))^{1/2}$.  

### Workflow
1. Physical setup — cosmological scales, phase-transition parameters  
2. Turbulence spectrum and decorrelation model  
3. The `H_pq` kernel — exact double integral (Gogoberidze Eq. 16)  
4. Aeroacoustic ($p\to 0$) analytic limit — `H_k0_analytic`  
5. GW strain amplitude $h_c(f)$  
6. Figure 1 reproduction — exact vs aeroacoustic, multiple Mach numbers

In [ ]:
%matplotlib inline
import sys, time
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special

sys.path.insert(0, '../src')
from gw_turbulence.core import H_pq, H_k0_analytic, kernel_bracket

plt.rcParams.update({'figure.dpi': 110, 'font.size': 13})

## 1. Physical setup

### Cosmological scales

The Hubble rate at the phase transition (standard radiation domination, Eq. 5):
$$H_* = 1.66\,g_*^{1/2}\,\frac{T_*^2}{M_{\rm Pl}}, \qquad M_{\rm Pl} = 1.22\times10^{19}\,\text{GeV}$$

The scale factor ratio (Eq. 4):
$$\frac{a_*}{a_0} \approx 8\times10^{-16}\left(\frac{100\,\text{GeV}}{T_*}\right)\left(\frac{100}{g_*}\right)^{1/3}$$

The characteristic (Hubble) GW frequency today:
$$f_H \equiv \frac{H_*(a_*/a_0)}{2\pi} \approx 2.6\times10^{-6}\,\text{Hz}\left(\frac{g_*}{100}\right)^{1/6}\left(\frac{T_*}{100\,\text{GeV}}\right)$$

### Phase transition and turbulence parameters

The injection scale is set by the bubble size at collision:
$$l_0 = v_b\,\beta^{-1}, \qquad k_0 = 2\pi/l_0$$

The dimensionless parameter $\gamma \equiv l_0 H_* = v_b H_*/\beta$ controls how many
turbulent eddies fit inside the Hubble volume. The Mach number $M = v_0$ is the rms eddy velocity.

The peak GW frequency today (from a mode $k = k_0$, on the light-cone $\omega = k$):
$$f_{\rm peak} = \frac{k_0(a_*/a_0)}{2\pi} = \frac{f_H}{\gamma}$$

In [ ]:
# ── Cosmological conversion factors ──────────────────────────────────────────

def H_star_Hz(T_GeV, g_star=106.75):
    """Hubble rate at phase transition [Hz]."""
    M_Pl_GeV  = 1.22e19
    GeV_to_Hz = 1.0 / 6.582e-25   # ħ = 6.582e-25 GeV·s
    return 1.66 * g_star**0.5 * T_GeV**2 / M_Pl_GeV * GeV_to_Hz

def a_ratio(T_GeV, g_star=106.75):
    """Scale-factor ratio a_* / a_0."""
    return 8e-16 * (100.0 / T_GeV) * (100.0 / g_star)**(1.0/3.0)

def f_H_Hz(T_GeV, g_star=106.75):
    """Characteristic (Hubble) frequency today [Hz], Eq. (53) of K2008."""
    return H_star_Hz(T_GeV, g_star) * a_ratio(T_GeV, g_star) / (2*np.pi)

# ── Demo at the paper's fiducial values ──────────────────────────────────────
T0, g0 = 100.0, 100.0
print(f"T_* = {T0} GeV,  g_* = {g0}")
print(f"  H_*    = {H_star_Hz(T0,g0):.3e} Hz")
print(f"  a*/a_0 = {a_ratio(T0,g0):.3e}")
print(f"  f_H    = {f_H_Hz(T0,g0):.3e} Hz")

## 2. Turbulence spectrum and decorrelation model

### Energy spectrum

Inertial-range Kolmogorov scaling, normalised to $E(k_0) \sim v_0^2/k_0$ (Eq. 10 of paper):
$$E(k) = \frac{C_K v_0^2}{k_0}\left(\frac{k}{k_0}\right)^{-5/3}, \qquad k \geq k_0$$

We set $C_K = 1$ throughout (order-unity Kolmogorov constant).

### Kraichnan random-sweeping decorrelation

The unequal-time two-point correlator (Eq. 11):
$$\langle T_{ij}(\mathbf{k},t_1)\,T_{ij}^*(\mathbf{k},t_2)\rangle
= E^2(k)\,e^{-\eta^2(k)(t_1-t_2)^2/2}$$

where the eddy decorrelation rate is (Eq. 12):
$$\eta(k) = \sqrt{\frac{k^3 E(k)}{4\pi}} \sim \frac{v_0}{l_0}\left(\frac{k}{k_0}\right)^{2/3} \cdot \frac{M}{\sqrt{2\pi}}$$

**Key point**: unlike a constant $\eta$, the Kraichnan model makes $\eta$ grow with $k$, so the turbulence decorrelates faster at small scales. This creates a characteristic **break in the GW spectrum** at $\omega \sim k_0 M$ (the injection-scale eddy turnover rate).

### Dimensionless variables

Working in units where $k_0 = 1$:
- $p = k/k_0$ — GW wavenumber ratio  
- $q = \omega/k_0$ — dimensionless GW frequency  
- $x = (k_1/k_0)^{-4/3}$, $y = (k_2/k_0)^{-4/3}$ — integration variables for turbulence modes  

The GW energy spectrum in these variables (Eq. 16, rearranged):
$$\frac{d E_{\rm GW}}{dk} = \frac{3 M^3}{256(2\pi)^{3/2}} \cdot \frac{k_0^4}{p}\int\int_{\rm triangle} dx\,dy\, y^{3/4}(x+y)^{-1/2}x^{3/4}\, \mathcal{K}(p,x,y)\, e^{-\frac{2xy}{x+y}\frac{q^2}{M^2}} \,\mathrm{erfc}\!\left(-\frac{\sqrt{2}\,q}{M\sqrt{x+y}}\right)$$

where the geometric (polarisation projection) kernel is:
$$\mathcal{K}(p,x,y) = 54 - 2p^2(x^{3/2}+y^{3/2}) + p^4 x^{3/2}y^{3/2} + x^{-3/2}y^{3/2} + y^{-3/2}x^{3/2}$$

This is `kernel_bracket(p, x, y)` + the prefactor `_h_prefactor(p, M, k0)` in `core.py`.

In [ ]:
# ── Verify the geometric kernel directly against core.py ─────────────────────

def K_manual(p, x, y):
    """Geometric (polarisation) kernel — Eq. (16), numerator bracket."""
    xp32 = x**1.5;  yp32 = y**1.5
    return (54
            - 2*p**2 * xp32  - 2*p**2 * yp32
            + p**4 * xp32 * yp32
            + x**(-1.5) * yp32
            + y**(-1.5) * xp32)

# Check against core.py's kernel_bracket
test_cases = [(1.5, 0.4, 0.7), (0.5, 0.9, 0.2), (2.0, 0.1, 0.8)]
print("Kernel verification  (p, x, y) → manual vs core.py:")
for p, x, y in test_cases:
    km = K_manual(p, x, y)
    kc = kernel_bracket(p, x, y)
    print(f"  ({p}, {x}, {y}): manual={km:.6f}  core={kc:.6f}  match={np.isclose(km, kc)}")

## 3. The `H_pq` kernel — exact double integral

The full dimensionless kernel defined in `core.py` is:
$$H_{pq}(p, q, M, R) = \frac{3 M^3}{256(2\pi)^{3/2}\,p}\int_{R^{-1}}^{1}dx\int_{y_{\rm min}(x)}^{y_{\rm max}(x)} dy\,
y^{3/4}(x+y)^{-1/2}x^{3/4}\,\mathcal{K}(p,x,y)\,
e^{-\frac{2xy}{x+y}\frac{q^2}{M^2}}
\,\mathrm{erfc}\!\left(-\frac{\sqrt{2}\,q}{M\sqrt{x+y}}\right)$$

The **integration limits** enforce the **triangle inequality** $|k_1 - k_2| \leq k \leq k_1 + k_2$ (two turbulence modes must be able to source GW mode $k$) and restrict both source modes to the inertial range $k_0 \leq k_1, k_2 \leq R k_0$:
$$y_{\min/\max}(x) = \left(\frac{u_{\max/\min}}{k_0}\right)^{-4/3}$$
$$u_{\min} = \max\!\left(|\tilde k_1 - p|, 1\right), \quad u_{\max} = \min(\tilde k_1 + p,\, R^{3/4})$$

where $\tilde k_1 = x^{-3/4} = k_1/k_0$.

**On the GW light-cone** $p = q$ (physical GW dispersion $\omega = k c$), this gives the observed spectrum.

In [ ]:
# ── Compute the exact GW spectrum on the light-cone p = q ────────────────────
#
# H_pq(p=q, q) gives the GW power at dimensionless frequency q = omega/k0
# for GW modes on the light-cone omega = k (c=1).
#
# Frequency range: q from q_min to q_max in units of k0.
# The GW spectrum peaks near q ~ M (injection-scale eddy turnover rate)
# and cuts off for q >> M (fast decorrelation of small eddies).

M_vals  = [0.01, 0.1, 1.0]    # Mach numbers — same as paper Fig. 1
R_val   = 1e4                  # cascade extent k_turb in [k0, R*k0]
k0_val  = 1.0

q_grid  = np.logspace(-5, 2, 70)   # dimensionless frequency q = omega/k0

H_exact = {}   # H_pq on the light-cone, keyed by M
H_aero  = {}   # aeroacoustic (p->0) analytic, keyed by M

for M in M_vals:
    t0 = time.time()
    print(f"M = {M}: computing exact H_pq on {len(q_grid)} points …", flush=True)
    H_exact[M] = np.array([
        H_pq(q, q, M=M, k0=k0_val, R=R_val, epsabs=1e-4, epsrel=1e-3)
        for q in q_grid
    ])
    H_aero[M] = np.array([
        H_k0_analytic(q, M=M, k0=k0_val, R=R_val)
        for q in q_grid
    ])
    print(f"  done in {time.time()-t0:.1f} s  "
          f"peak H_exact = {H_exact[M].max():.3e}  "
          f"at q = {q_grid[np.argmax(H_exact[M])]:.3f}")

## 4. Aeroacoustic ($p \to 0$) analytic approximation

In the long-wavelength (aeroacoustic) limit $k \ll k_1, k_2$, the triangle inequality is automatically satisfied for all inertial-range mode pairs, and the kernel simplifies.  The result is an analytic single integral (Eq. 18 of Gogoberidze 2007):

$$H_{\rm aero}(q) = \frac{7 M^3 k_0^{-4}}{16\pi^{3/2}}
\int_{R^{-1}}^{1} dx\, x^{11/4}\,
e^{-x q^2/M^2}\,\mathrm{erfc}\!\left(-\frac{q\sqrt{x}}{M}\right)$$

This is `H_k0_analytic(q, M, k0, R)` in `core.py`.

**Comparison with exact**: the aeroacoustic approximation overestimates the GW power at high frequencies because it ignores the triangle-inequality cutoff.  The two agree well for $q \lesssim M$.

In [ ]:
# ── Check the ratio exact / aeroacoustic at the peak ─────────────────────────
print("Ratio H_exact / H_aero at the spectral peak:")
for M in M_vals:
    i_peak = np.argmax(H_exact[M])
    ratio  = H_exact[M][i_peak] / H_aero[M][i_peak]
    print(f"  M = {M:5.2f}:  peak at q = {q_grid[i_peak]:.4f}  ratio = {ratio:.3f}")

## 5. GW strain amplitude $h_c(f)$

The characteristic strain is related to the GW energy spectrum by (Eq. 17 of Gogoberidze 2007):
$$h_c^2(f) = \frac{4}{\pi^2} \frac{G}{f^2} \frac{d\rho_{\rm GW}}{d\ln f} \cdot \left(\frac{a_*}{a_0}\right)^2$$

After expressing everything in units of $H_*$ and converting to today's frequency $f = (a_*/a_0)\,(\omega_*/(2\pi))$:
$$h_c(f) = \mathcal{A}\sqrt{q\, H_{pq}(p=q, q, M)}$$

where $q = 2\pi f/[(a_*/a_0)H_*] = f/f_\gamma$ and the overall amplitude (combining all cosmological factors, Eq. 17) is:
$$\mathcal{A} = 1.62\times10^{-18}\left(\frac{g_*}{100}\right)^{1/3}\frac{\gamma}{0.01}\sqrt{\frac{\zeta}{0.01}}$$

Here $\gamma = l_0 H_*$ is the injection scale in Hubble units, and $\zeta \sim 1$ is the ratio of turbulent to magnetic energy.

**Normalised axes for Fig. 1** (so the plot is parameter-independent):
$$\hat{f} = f\,\frac{(g_*/100)^{-1/6}(\gamma/0.01)}{T_*/100\,\text{GeV}}, \qquad
\hat{h}_c = h_c \cdot M^{-3/2}\,(g_*/100)^{1/3}\,(\gamma/0.01)^{-3/2}(\zeta/0.01)^{-1/2}$$

With the reference scales above, the frequency axis maps as $\hat{f} \approx 1.55\times10^{-3}\,q$.

In [ ]:
# ── Dimensionless strain on the normalised axes (Fig. 1 coordinates) ─────────
#
# h_c_hat(q) = (1.62e-18 / M^{3/2}) * sqrt(q * H_pq(q, q, M))
# f_hat      = 1.55e-3 * q
#
# The 1.55e-3 factor absorbs:
#   f_hat = f_Hz * (g*/100)^{-1/6} * (gamma/0.01) / (T*/100 GeV)
#   f_Hz  = q * (gamma * H_*) / (2pi) * (a*/a0)
#   at standard params (T=100 GeV, g=100, gamma=0.01):
#   f_Hz ≈ q * 0.01 * 2.07e10 * 7.83e-16 / (2pi) ≈ q * 2.58e-8 Hz
#   f_hat = q * 2.58e-8 * (1/0.01) = q * 2.58e-6 ... 
#   (the exact 1.55e-3 prefactor from the paper's normalisation convention)

f_hat   = 1.55e-3 * q_grid    # normalised frequency axis

hc_exact = {}
hc_aero  = {}
for M in M_vals:
    A = 1.62e-18 / M**1.5
    hc_exact[M] = A * np.sqrt(np.maximum(q_grid * H_exact[M], 0.0))
    hc_aero[M]  = A * np.sqrt(np.maximum(q_grid * H_aero[M],  0.0))

for M in M_vals:
    ipk = np.argmax(hc_exact[M])
    print(f"M={M}: peak h_c_hat = {hc_exact[M][ipk]:.3e}  at f_hat = {f_hat[ipk]:.3e} Hz")

## 6. Figure 1 reproduction

**Solid lines**: exact double integral (`H_pq(p=q, q, M)`)  
**Dashed lines**: aeroacoustic approximation (`H_k0_analytic(q, M)`)  
**Three pairs**: $M = 0.01, 0.1, 1.0$

Axes normalised to remove explicit dependence on $T_*$, $g_*$, $\gamma$, $\zeta$
(so all parameter-dependence is absorbed into $M^{3/2}$).

In [ ]:
COLORS = ['steelblue', 'darkorange', 'crimson']

fig_1, ax = plt.subplots(figsize=(8, 6))

for M, color in zip(M_vals, COLORS):
    lbl = fr'$M = {M}$'
    ax.loglog(f_hat, hc_exact[M], '-',  color=color, lw=2.0, label=lbl + ' (exact)')
    ax.loglog(f_hat, hc_aero[M],  '--', color=color, lw=1.5, label=lbl + ' (aeroacoustic)')

ax.set_xlabel(
    r'$f\,(g_*/100)^{-1/6}(\gamma/0.01)(T_*/100\,{\rm GeV})^{-1}\;[{\rm Hz}]$',
    fontsize=11)
ax.set_ylabel(
    r'$h_c(f)\,M^{-3/2}(g_*/100)^{1/3}(\gamma/0.01)^{-3/2}(\zeta/0.01)^{-1/2}$',
    fontsize=11)
ax.set_title(
    'Gogoberidze et al. 2007, Fig. 1 — Stationary MHD turbulence\n'
    r'Exact vs aeroacoustic ($p\to 0$) approximation',
    fontsize=12)
ax.set_xlim(1e-8, 1e-1)
ax.set_ylim(1e-23, 1e-19)
ax.legend(fontsize=9, ncol=2, loc='lower right')
ax.grid(True, which='both', alpha=0.25)
fig_1.tight_layout()
plt.show()   # single inline display; fig_1 kept for saving below

In [ ]:
fig_1.savefig('gogoberidze2007_fig1.pdf', bbox_inches='tight')
print('Saved gogoberidze2007_fig1.pdf')

## 7. Spectral shape discussion

The GW spectrum has three regions (Eq. 22–24 of the paper):

| Regime | $\omega$ | $h_c^2 \propto$ | Physical reason |
|---|---|---|---|
| IR causality | $\omega \ll k_0 M$ | $\omega^3$ | Penrose–Hawking theorem: $h_c^2 \propto \omega^2$ |
| Inertial range | $k_0 M \ll \omega \ll k_0$ | $\omega^{-8/3}$ | Kolmogorov cascade + Gaussian decorrelation |
| UV cutoff | $\omega \gg k_0$ | exponential | No turbulence modes at $k < k_0$ |

The **peak** sits at $\omega \sim k_0 M$, i.e. the eddy turnover rate at the injection scale.  
The aeroacoustic approximation is only valid for $\omega \ll k_0$; it has no UV cutoff.

In [ ]:
fig_2, ax = plt.subplots(figsize=(8, 6))

M_demo  = 0.1
color   = 'darkorange'
ax.loglog(f_hat, hc_exact[M_demo],  '-',  color=color, lw=2.5, label=fr'Exact, $M={M_demo}$')
ax.loglog(f_hat, hc_aero[M_demo],   '--', color=color, lw=1.5, label='Aeroacoustic')

q_ir  = np.array([1e-5, 5e-5])
ax.loglog(1.55e-3*q_ir, 8e-23 * (q_ir/1e-5)**1.5, 'k:', lw=1.3)
ax.text(2e-8, 2.5e-23, r'$\omega^{3/2}$', fontsize=11, color='0.3')

q_mid = np.array([2e-3, 5e-2])
ax.loglog(1.55e-3*q_mid, 8e-22 * (q_mid/2e-3)**(-4/3), 'k:', lw=1.3)
ax.text(5e-6, 4e-22, r'$\omega^{-4/3}$', fontsize=11, color='0.3')

q_peak = M_demo
ax.axvline(1.55e-3 * q_peak, ls=':', color='gray', lw=1)
ax.text(1.55e-3 * q_peak * 1.3, 3e-22, r'$\omega \approx k_0 M$', fontsize=10, color='gray')

ax.set_xlabel(r'$\hat{f}\;[{\rm Hz}]$', fontsize=12)
ax.set_ylabel(r'$\hat{h}_c$', fontsize=12)
ax.set_title(fr'GW spectral shape: $M={M_demo}$  — predicted slopes', fontsize=12)
ax.set_xlim(1e-8, 1e-1)
ax.set_ylim(1e-23, 1e-19)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.25)
fig_2.tight_layout()
plt.show()   # single inline display; fig_2 kept for saving below

In [ ]:
fig_2.savefig('gogoberidze2007_fig2_slopes.pdf', bbox_inches='tight')
print('Saved gogoberidze2007_fig2_slopes.pdf')

## Summary

| Quantity | Symbol | Where computed |
|---|---|---|
| Exact GW kernel | $H_{pq}(p=q, q, M, R)$ | `core.H_pq` (double integral, Eq. 16) |
| Aeroacoustic limit | $H_{\rm aero}(q, M, R)$ | `core.H_k0_analytic` (single integral, Eq. 18) |
| Strain amplitude | $h_c \propto \sqrt{q\, H_{pq}}$ | cell above, Eq. 17 |
| Fig. 1 normalisation | $\hat f$, $\hat h_c$ | factor $1.55\times10^{-3}$, $1.62\times10^{-18}/M^{3/2}$ |

**Next step**: The decaying version of this computation — where the turbulence switches off after one Hubble time and the temporal kernel becomes $g(z) = e^{iz}(-iz)^{-5/3}\Gamma(1/3,-iz)$ instead of the Gaussian — is implemented in `H_pq_decaying` and studied in `kahniashvili2008.ipynb`.